# DiSCo Python Test
Dieses Notebook testet die übersetzte Python-Version (`disco.py`) mit dem `dube` Datensatz, der auch in der R-Vignette (`Dube2019.Rmd`) verwendet wird.

In [1]:
import pandas as pd
import numpy as np
import pyreadr # Zum Einlesen von .rda Dateien
import sys
import os
import matplotlib.pyplot as plt

# Fasse den übergeordneten Ordner ins Visier, damit Python das "python/" Verzeichnis als Modul (Package) erkennt
sys.path.append(os.path.abspath('..'))

# Jetzt können wir es sicher als Modul importieren
np.random.seed(999)  # Für Reproduzierbarkeit
from python.disco import DiSCo

## 1. Dube (2019) Daten laden
Wir laden die Daten aus dem R-Package `data/dube.rda`. Laut der Vignette wird `state=2` als Treatment betrachtet und `t0=2003`.

In [2]:
# Lade die originale R-Datei
result = pyreadr.read_r('../data/dube.rda')
df_raw = result['dube']

# Das R-Package erwartet die Spalten id_col, time_col, y_col.
# In dube.rda sind die Spalten fips (state), year und adj0contpov (income).
df = df_raw.copy()
df = df.rename(columns={
    'fips': 'id_col',
    'year': 'time_col',
    'adj0contpov': 'y_col'
})

print(f"Datensatz geladen: {len(df)} Zeilen")
df.head()

Datensatz geladen: 652870 Zeilen


,time_col,id_col,y_col
0,1998.0,1.0,2.791217
1,1998.0,1.0,0.165951
2,1998.0,1.0,1.674730
3,1998.0,1.0,2.088006
4,1998.0,1.0,3.639715


In [3]:
df['id_col'].unique()

array([ 1.,  2.,  4.,  5.,  8., 13., 16., 18., 19., 20., 21., 22., 24.,
       26., 28., 29., 30., 31., 32., 33., 35., 37., 38., 39., 40., 42.,
       45., 46., 47., 48., 49., 51., 54., 56.])

## 2. DiSCo Modell initialisieren und fitten
Wir rufen nun unser `DiSCo`-Modell auf. Für diesen Test verwenden wir eine geringere Anzahl an Ziehungen (`M=100`) und 1 CPU-Kern (`num_cores=1`), da der Datensatz sehr klein ist.

In [ ]:
print("Initialisiere DiSCo mit Dube-Daten...")
disco_model = DiSCo(
    df=df,
    id_col='id_col',
    time_col='time_col',
    y_col='y_col',
    id_col_target=2,     # FIPS = 2 (Alaska) als Treatment
    t0=2003,             # Treatment im Jahr 2003
    M=1000,              # Monte Carlo Draws 
    G=1000,              # Grid Size
    num_cores=1,        # Alle CPU-Kerne für maximalen Speed nutzen
    simplex=True,        # Entspricht den Parametern in der Vignette
    q_max=0.9            # Entspricht den Parametern in der Vignette
)

weights = disco_model.fit()

# Ausgabe der wichtigsten 5 Gewichte (ungleich null)
weight_series = pd.Series(weights, index=disco_model.controls_id)
print("\nHöchstgewichtete Kontroll-Einheiten:")
print(weight_series[weight_series > 0.01].sort_values(ascending=False).head(5))

Initialisiere DiSCo mit Dube-Daten...


In [16]:
print(weight_series[weight_series > 0.01].sort_values(ascending=False).head(10))

39.0    0.156919
26.0    0.114723
45.0    0.092852
51.0    0.088746
20.0    0.087148
24.0    0.058267
37.0    0.057599
13.0    0.049505
29.0    0.042507
4.0     0.034955
dtype: float64
